# EstateIQ — Property Image Classification Using CNN

This notebook documents the corrected five-class Room Photo Checker training workflow. It keeps the EstateIQ interface unchanged and focuses on image quality, deterministic evaluation, label order, and model preprocessing.

**Classes (fixed output order):** Bathroom, Bedroom, Dinning, Kitchen, Livingroom.

**Input contract:** RGB image, resized to 96 × 96; pixel values are passed in the 0–255 range. The model itself applies 1/255 scaling.

The independent test set is not used for training or early stopping. Validation and test datasets are not shuffled, so predictions stay aligned with their true labels.


In [ ]:
from pathlib import Path
from collections import defaultdict, Counter
import hashlib, json, random
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

SEED = 42
IMG_SIZE = (96, 96)
BATCH_SIZE = 32
EPOCHS = 24
CLASS_NAMES = ["Bathroom", "Bedroom", "Dinning", "Kitchen", "Livingroom"]
CLASS_TO_INDEX = {name: i for i, name in enumerate(CLASS_NAMES)}
VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
ROOT = Path.cwd()
if not (ROOT / "House_Room_Dataset").is_dir():
    ROOT = ROOT / "Property Image Classification Using CNN"
DATA_DIR = ROOT / "House_Room_Dataset"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_PATH = ROOT / "property_cnn_model.keras"

random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))


## 1. Dataset integrity and duplicate audit

The supplied data contains 5,250 image files. The audit below removes exact repeat files within a class and excludes exact duplicate images that appear under conflicting class-folder labels. It also records unreadable files. This does not automatically relabel visually ambiguous photos; those still need human review.


In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

hash_groups = defaultdict(list)
unreadable = []
for class_name in CLASS_NAMES:
    folder = DATA_DIR / class_name
    if not folder.is_dir():
        raise FileNotFoundError(folder)
    for path in sorted(folder.rglob("*")):
        if not path.is_file() or path.suffix.lower() not in VALID_EXTS:
            continue
        try:
            with Image.open(path) as image:
                image.verify()
            hash_groups[sha256_file(path)].append((path, class_name))
        except Exception as exc:
            unreadable.append({"file": str(path.relative_to(ROOT)), "error": str(exc)})

records, duplicate_rows = [], []
for digest, group in sorted(hash_groups.items()):
    labels = sorted({label for _, label in group})
    if len(labels) > 1:
        duplicate_rows.append({"sha256": digest, "action": "excluded_conflicting_class_duplicate",
                               "files": " | ".join(str(p.relative_to(ROOT)) for p, _ in group),
                               "labels": " | ".join(labels)})
        continue
    group.sort(key=lambda item: str(item[0]).casefold())
    records.append(group[0])
    if len(group) > 1:
        duplicate_rows.append({"sha256": digest, "action": "kept_one_same_class_duplicate",
                               "files": " | ".join(str(p.relative_to(ROOT)) for p, _ in group),
                               "labels": labels[0]})

pd.DataFrame(duplicate_rows).to_csv(OUTPUT_DIR / "duplicate_audit.csv", index=False)
(OUTPUT_DIR / "unreadable_files.json").write_text(json.dumps(unreadable, indent=2), encoding="utf-8")
counts = Counter(label for _, label in records)
print("Raw image files:", sum(counts.values()) + sum(len(x) for x in unreadable))
print("Usable unique images:", len(records))
print("Class counts:", dict(counts))
print("Duplicate groups audited:", len(duplicate_rows), "Unreadable:", len(unreadable))


## 2. Fixed stratified train / validation / test split

A 70/15/15 split is used. Only the validation split controls early stopping. The test split is held back until the final evaluation.


In [ ]:
paths = np.array([str(path) for path, _ in records])
labels = np.array([CLASS_TO_INDEX[label] for _, label in records], dtype=np.int32)
indices = np.arange(len(records))
train_idx, hold_idx = train_test_split(indices, test_size=0.30, stratify=labels, random_state=SEED)
val_idx, test_idx = train_test_split(hold_idx, test_size=0.50, stratify=labels[hold_idx], random_state=SEED + 1)

train_paths, train_labels = paths[train_idx], labels[train_idx]
val_paths, val_labels = paths[val_idx], labels[val_idx]
test_paths, test_labels = paths[test_idx], labels[test_idx]

print("Train / validation / test:", len(train_idx), len(val_idx), len(test_idx))
print("Training class counts:", dict(zip(CLASS_NAMES, np.bincount(train_labels, minlength=len(CLASS_NAMES)))))

split_df = pd.concat([
    pd.DataFrame({"split": split, "file": p, "label": [CLASS_NAMES[i] for i in y]})
    for split, p, y in [("train", train_paths, train_labels), ("validation", val_paths, val_labels), ("test", test_paths, test_labels)]
], ignore_index=True)
split_df.to_csv(OUTPUT_DIR / "data_split.csv", index=False)


## 3. Build deterministic TensorFlow datasets

Only the training pipeline is shuffled and augmented. Validation/test stay ordered. Images are kept in the original 0–255 scale because the model includes a `Rescaling(1/255)` layer.


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def decode_and_resize(path, label):
    raw = tf.io.read_file(path)
    image = tf.io.decode_image(raw, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMG_SIZE, antialias=True)
    return image, label

def augment_image(image, label):
    image = tf.image.random_flip_left_right(image, seed=SEED)
    image = tf.image.random_brightness(image, max_delta=18.0, seed=SEED)
    image = tf.image.random_contrast(image, lower=0.90, upper=1.10, seed=SEED)
    image = tf.clip_by_value(image, 0.0, 255.0)
    return image, label

def make_dataset(file_paths, file_labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((file_paths, file_labels))
    if training:
        ds = ds.shuffle(len(file_paths), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(decode_and_resize, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(augment_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(train_paths, train_labels, training=True)
val_ds = make_dataset(val_paths, val_labels, training=False)
test_ds = make_dataset(test_paths, test_labels, training=False)

for batch_images, batch_labels in train_ds.take(1):
    print("Batch shape:", batch_images.shape, "Labels shape:", batch_labels.shape)


## 4. CNN architecture

The updated network is a compact residual CNN with batch normalization, swish activations, global average pooling, and dropout. Its input scaling and 96×96 RGB contract match the saved application model.


In [ ]:
def residual_block(tensor, filters, stride=1, name="block"):
    shortcut = tensor
    conv_input = layers.ZeroPadding2D(1, name=f"{name}_pad")(tensor) if stride == 2 else tensor
    conv_padding = "valid" if stride == 2 else "same"
    x = layers.Conv2D(filters, 3, strides=stride, padding=conv_padding, use_bias=False, name=f"{name}_conv1")(conv_input)
    x = layers.BatchNormalization(epsilon=1e-5, name=f"{name}_bn1")(x)
    x = layers.Activation("swish", name=f"{name}_act1")(x)
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False, name=f"{name}_conv2")(x)
    x = layers.BatchNormalization(epsilon=1e-5, name=f"{name}_bn2")(x)
    if stride != 1 or int(shortcut.shape[-1]) != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride, padding="same", use_bias=False, name=f"{name}_shortcut")(shortcut)
        shortcut = layers.BatchNormalization(epsilon=1e-5, name=f"{name}_shortcut_bn")(shortcut)
    x = layers.Add(name=f"{name}_add")([x, shortcut])
    return layers.Activation("swish", name=f"{name}_out")(x)

inputs = keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3), name="room_image")
x = layers.Rescaling(1.0 / 255.0, name="pixel_scaling")(inputs)
x = layers.ZeroPadding2D(1, name="stem_pad")(x)
x = layers.Conv2D(16, 3, strides=2, padding="valid", use_bias=False, name="stem_conv")(x)
x = layers.BatchNormalization(epsilon=1e-5, name="stem_bn")(x)
x = layers.Activation("swish", name="stem_act")(x)
x = residual_block(x, 16, 1, "res16")
x = residual_block(x, 32, 2, "res32")
x = residual_block(x, 64, 2, "res64")
x = residual_block(x, 96, 2, "res96")
x = layers.GlobalAveragePooling2D(name="global_average_pool")(x)
x = layers.Dense(128, activation="swish", name="classifier_dense")(x)
x = layers.Dropout(0.40, name="classifier_dropout")(x)
outputs = layers.Dense(len(CLASS_NAMES), activation="softmax", name="room_probabilities")(x)
model = keras.Model(inputs, outputs, name="EstateIQ_Room_CNN_v2")
model.summary()


In [ ]:
train_counts = np.bincount(train_labels, minlength=len(CLASS_NAMES))
class_weights = np.sqrt(train_counts.sum() / (len(CLASS_NAMES) * train_counts)).astype(np.float32)
weight_tensor = tf.constant(class_weights, dtype=tf.float32)
train_weighted_ds = train_ds.map(lambda image, label: (image, label, tf.gather(weight_tensor, label)))

model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=0.001, weight_decay=1e-4, clipnorm=1.0),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
)
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.4, patience=2, min_lr=1e-5, verbose=1),
    keras.callbacks.ModelCheckpoint(str(MODEL_PATH), monitor="val_loss", save_best_only=True, verbose=1),
]
history = model.fit(train_weighted_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)


## 5. Independent test evaluation

The report below is calculated from a single ordered pass over the untouched test dataset. This avoids the earlier notebook issue where iterating a shuffled validation dataset separately for `y_true` and predictions could misalign labels.


In [ ]:
best_model = keras.models.load_model(MODEL_PATH, compile=False)
test_images = test_ds.map(lambda image, label: image)
y_prob = best_model.predict(test_images, verbose=1)
y_true = np.concatenate([labels.numpy() for _, labels in test_ds])
y_pred = np.argmax(y_prob, axis=1)

print(classification_report(y_true, y_pred, labels=list(range(len(CLASS_NAMES))),
                            target_names=CLASS_NAMES, digits=3, zero_division=0))
print("Test accuracy:", round(accuracy_score(y_true, y_pred), 4))
print("Macro F1:", round(f1_score(y_true, y_pred, average="macro", zero_division=0), 4))

report = classification_report(y_true, y_pred, labels=list(range(len(CLASS_NAMES))),
                               target_names=CLASS_NAMES, output_dict=True, zero_division=0)
pd.DataFrame(report).transpose().to_csv(OUTPUT_DIR / "test_classification_report.csv")
pd.DataFrame(confusion_matrix(y_true, y_pred, labels=list(range(len(CLASS_NAMES)))),
             index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(OUTPUT_DIR / "test_confusion_matrix.csv")


In [ ]:
# Save a prediction audit file for checking individual mistakes.
rows = []
for i, (path, true_idx, pred_idx, score) in enumerate(zip(test_paths, y_true, y_pred, y_prob.max(axis=1))):
    rows.append({"file": path, "actual": CLASS_NAMES[int(true_idx)],
                 "predicted": CLASS_NAMES[int(pred_idx)], "model_score": float(score),
                 "correct": bool(true_idx == pred_idx)})
pd.DataFrame(rows).to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)
print("Saved model:", MODEL_PATH)


## 6. Results from the packaged retrained model

The currently packaged model was trained for 12 epochs with early stopping; the best validation checkpoint was epoch 7. Its independent 777-image test set achieved:

- **Accuracy:** 53.7%
- **Macro F1:** 0.531
- **Weighted F1:** 0.526

| Room | Precision | Recall | F1 | Test samples |
|---|---:|---:|---:|---:|
| Bathroom | 0.469 | 0.744 | 0.575 | 90 |
| Bedroom | 0.573 | 0.317 | 0.408 | 186 |
| Dining room | 0.640 | 0.696 | 0.667 | 171 |
| Kitchen | 0.551 | 0.415 | 0.474 | 142 |
| Living room | 0.475 | 0.601 | 0.531 | 188 |

The old model scored 41.1% on the same fixed test split. The updated model improved overall accuracy and raised Living room recall from 22.5% to 60.1%. This is a meaningful improvement, but 53.7% is still moderate performance—not a production-grade classifier. Softmax scores are not calibrated probabilities. Ambiguous open-plan rooms and residual label noise remain limitations.
